# Competitive-binding Kd models

This notebook demonstrates competitive-binding models: `comp_3st_specific`, `comp_3st_total`, `comp_4st_specific`, and `comp_4st_total`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bindcurve as bc

rng = np.random.default_rng(123)

## Helper for synthetic competition data

In [ ]:
def make_competition_data(model_name, params, compound_id='competitor_a'):
    model = bc.get_model(model_name)
    concentrations = np.logspace(-3, 2, 22)
    rows = []
    for experiment_id, scale in {'exp1': 0.95, 'exp2': 1.00, 'exp3': 1.05}.items():
        for concentration in concentrations:
            response = model.evaluate(np.array([concentration * scale]), **params)[0]
            for replicate_id in range(1, 4):
                rows.append({
                    'compound_id': compound_id,
                    'experiment_id': experiment_id,
                    'concentration': concentration,
                    'replicate_id': f'rep{replicate_id}',
                    'response': response + rng.normal(0, 0.75),
                })
    return bc.DoseResponseData.from_dataframe(pd.DataFrame(rows), concentration_unit='uM', response_unit='percent')

## Three-state specific competition

In [ ]:
params_3st = {
    'ymin': 5.0, 'ymax': 95.0,
    'RT': 0.05, 'LsT': 0.005, 'Kds': 0.02,
    'Kd': 1.6,
}
data_3st = make_competition_data('comp_3st_specific', params_3st)
results_3st = bc.fit(
    data_3st,
    model='comp_3st_specific',
    fixed={'ymin': 5.0, 'ymax': 95.0, 'RT': 0.05, 'LsT': 0.005, 'Kds': 0.02},
)
results_3st.summary_to_dataframe()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bc.plot_curves(data_3st, results_3st, ax=ax, confidence_band=True)
ax.legend()
plt.show()

## Three-state total competition

The total/nonspecific model uses dimensionless `N`.

In [ ]:
params_3st_total = {
    'ymin': 4.0, 'ymax': 90.0,
    'RT': 0.05, 'LsT': 0.005, 'Kds': 0.02,
    'N': 0.35,
    'Kd': 2.4,
}
data_3st_total = make_competition_data('comp_3st_total', params_3st_total)
results_3st_total = bc.fit(
    data_3st_total,
    model='comp_3st_total',
    fixed={'ymin': 4.0, 'ymax': 90.0, 'RT': 0.05, 'LsT': 0.005, 'Kds': 0.02, 'N': 0.35},
)
results_3st_total.summary_to_dataframe()

## Four-state specific competition

The four-state model adds `Kd3`, the affinity of labeled ligand to the receptor-competitor complex.

In [ ]:
params_4st = {
    'ymin': 5.0, 'ymax': 95.0,
    'RT': 0.05, 'LsT': 0.005, 'Kds': 0.02,
    'Kd3': 0.5,
    'Kd': 1.6,
}
data_4st = make_competition_data('comp_4st_specific', params_4st)
results_4st = bc.fit(
    data_4st,
    model='comp_4st_specific',
    fixed={'ymin': 5.0, 'ymax': 95.0, 'RT': 0.05, 'LsT': 0.005, 'Kds': 0.02, 'Kd3': 0.5},
)
results_4st.summary_to_dataframe()

## IC50-to-Kd conversion

Conversion is separate from fitting.

In [ ]:
bc.convert_ic50_to_kd(
    model='cheng_prusoff',
    IC50=10.0,
    LsT=0.005,
    Kds=0.02,
    unit='uM',
)